# GPT from Scratch — Little Shakespeare

**Task:** Implement and train a GPT-style Transformer on the Little Shakespeare dataset.  

This notebook is **modular**: each major component (Dataset, Embeddings, Attention, FFN, DecoderBlock, GPT, Training, Generation) is implemented as a separate class or function and explained below.

---

## 1. Setup

In [ ]:
# Colab: Runtime -> Change runtime type -> GPU (must use GPU or training is very slow)
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("WARNING: On Colab use Runtime -> Change runtime type -> T4 GPU for much faster training.")

## 2. Dataset — Little Shakespeare (Modular: Data Layer)

We use two modular pieces:

- **`get_vocab(text)`** — Builds the character-level vocabulary from the corpus: every unique character gets an index (`stoi`: char → id) and we keep the reverse mapping (`itos`: id → char). Vocab size is typically ~65 (letters, punctuation, newline). This avoids any "unknown" token.
- **`ShakespeareDataset`** — PyTorch `Dataset` that:
  - Takes raw text, `block_size` (128), and `stoi`.
  - Encodes the full text into token IDs.
  - In `__getitem__(idx)` returns **two** sequences: **input** = tokens `[idx : idx+block_size]`, **target** = tokens `[idx+1 : idx+block_size+1]` (same length, shifted by 1). This is exactly the next-token prediction setup: given input, we predict target.
  - Train/val: we use **90%** of the text for training and **10%** for validation (no shuffle across that split), so the model is evaluated on unseen Shakespeare.

In [ ]:
import urllib.request
import torch
from torch.utils.data import Dataset, DataLoader

# --- get_vocab: build character -> id (stoi) and id -> character (itos), vocab size ~65 ---
def get_vocab(text):
    chars = sorted(list(set(text)))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos

# --- ShakespeareDataset: chunks of block_size, target = input shifted by 1 (next-token prediction) ---
class ShakespeareDataset(Dataset):
    def __init__(self, text, block_size, stoi):
        self.block_size = block_size
        self.stoi = stoi
        self.data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
with urllib.request.urlopen(URL) as f:
    text = f.read().decode("utf-8")
stoi, itos = get_vocab(text)
vocab_size = len(stoi)
n = len(text)
split_idx = int(n * 0.9)
train_ds = ShakespeareDataset(text[:split_idx], block_size=128, stoi=stoi)
val_ds = ShakespeareDataset(text[split_idx:], block_size=128, stoi=stoi)
print(f"Vocab size: {vocab_size}, Train blocks: {len(train_ds)}, Val blocks: {len(val_ds)}")

## 3. Model — GPT Architecture (Modular: Each Block Explained)

The model is built from **separate classes** so We can see each piece clearly.

---

### 3.1 Embeddings (inside `GPT`)

- **Token embedding:** `nn.Embedding(vocab_size, d_model)` — turns each character ID into a dense vector of size `d_model`. So each character is represented by a learned vector.
- **Position embedding:** `nn.Embedding(block_size, d_model)` — gives each position in the sequence (0, 1, …, 127) a learned vector. Transformers have no built-in notion of order, so we add position so the model knows "cat sat" vs "sat cat".
- **Combined input:** `x = token_emb(idx) + pos_emb(pos)` — we add token and position embeddings and feed that into the rest of the network.

---

### 3.2 CausalSelfAttention (the heart of the decoder)

- **Query, Key, Value (Q, K, V):** From the same input `x` we compute three projections: `Q`, `K`, `V`. Each token has a query ("what am I looking for?"), a key ("what do I represent?"), and a value ("content to pass on"). Attention scores are `Q @ K^T` (dot product between queries and keys), scaled by `1/sqrt(d_k)` for stability.
- **Causal mask (tril):** We use a **lower-triangular mask** so that position `i` can only attend to positions `j <= i`. So the model cannot "cheat" by looking at future tokens — it only uses the past. Implemented by setting scores for `j > i` to `-inf` before softmax, so those weights become 0.
- **Multi-head attention:** Instead of one big attention matrix, we split `d_model` into `n_heads` heads. Each head runs attention independently (different Q/K/V per head), then we concatenate and project back. So the model can learn different types of relationships (e.g. previous token, subject-verb, etc.) in parallel.

---

### 3.3 FeedForward (FFN) — the "thinking" block

- **Formula:** `FFN(x) = W2 · GELU(W1·x + b1) + b2`. Two linear layers with **GELU** activation in between. Applied **per token** independently. This adds non-linearity and capacity after attention.

---

### 3.4 DecoderBlock — residual connections and LayerNorm

- **Residual connection:** We do `x = x + sublayer(x)` so the input is added back to the output. This keeps the "signal" from getting lost in deep networks and helps gradients flow.
- **LayerNorm:** We normalize the features (per token) before attention and before FFN. This keeps activations at a stable scale and makes training easier.
- **Order in one block:** `x = x + attn(LayerNorm(x))`, then `x = x + ffn(LayerNorm(x))`. So: LayerNorm → Attention → add residual → LayerNorm → FFN → add residual.

---

### 3.5 GPT — stacking and output

- **Stacking:** We repeat the same `DecoderBlock` **N times**. So the model has depth: early layers can learn local patterns, later layers can learn higher-level structure.
- **Final layer:** After all blocks we apply a final `LayerNorm` and then a linear layer `lm_head` that maps from `d_model` to `vocab_size`. That gives **logits for the next token** at each position. Loss is cross-entropy between those logits and the target tokens.

In [ ]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F

# ========== CausalSelfAttention: Q, K, V + causal (tril) mask + multi-head ==========
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.register_buffer("causal_mask", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))
        self.W_qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.W_qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)

# ========== FeedForward: two linear layers + GELU (per token) ==========
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))

# ========== DecoderBlock: Attention -> Add & Norm -> FFN -> Add & Norm (residual + LayerNorm) ==========
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, block_size, dropout=0.1):
        super().__init__()
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

# ========== GPT: Token + Position embeddings -> N x DecoderBlock -> final LayerNorm -> lm_head (next-token logits) ==========
class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, d_model=256, n_heads=8, n_layers=6, d_ff=1024, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, block_size, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, loss

block_size = 128
# Fast run: smaller model (change to d_model=256, n_layers=6, d_ff=1024 for full quality)
model = GPT(vocab_size=vocab_size, block_size=block_size, d_model=128, n_heads=4, n_layers=3, d_ff=512).to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

## 4. Training — Loss, Optimizer, Schedule

- **Loss:** Cross-entropy over the next token at every position (average over all positions in the batch). So we are directly optimizing next-token prediction.
- **Optimizer:** **AdamW** (Adam with decoupled weight decay) as required. We use `weight_decay=0.01` for better generalization.
- **Block size:** Fixed at **128** as required.
- **Learning rate schedule:** **Warmup** for the first 200 steps (LR goes from 0 to peak), then **cosine decay** down to 10% of the peak until the end of training. This avoids early instability and keeps updates useful later.
- **Gradient clipping:** We clip the gradient norm to **1.0** so that updates do not explode in deep networks.
- **Logging:** We print the **initial loss** at step 0 (should be ~4.17 for random init over ~65 chars) and then every 200 steps we report **train loss** and **validation loss** so We can see learning and check for overfitting. Checkpoints are saved every 500 steps.

**Proper data prep (Section 2):** We use 90% train / 10% val split, block size 128, and targets = input shifted by 1 for next-token prediction. The dataset is loaded and chunked before training.

**Faster run (e.g. for testing):** Set `max_steps = 500` or `1000` in the cell below to finish quickly; use `5000` for full training and lower final loss.

In [ ]:
def get_lr(step, warmup_steps, max_steps, base_lr, min_ratio=0.1):
    if step < warmup_steps:
        return base_lr * step / warmup_steps
    decay = 0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / max(1, max_steps - warmup_steps)))
    return min_ratio * base_lr + (1 - min_ratio) * base_lr * decay

batch_size = 64
max_steps = 200
warmup_steps = 200
lr = 3e-4
grad_clip = 1.0
eval_interval = 200

# DataLoader + AdamW optimizer; we will log train_loss and val_loss every eval_interval steps.
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
train_losses, val_losses = [], []

In [ ]:
# Training loop: forward -> loss -> backward -> clip grad -> step; log initial loss at step 0, val loss every eval_interval.
train_iter = iter(train_loader)
for step in range(max_steps):
    try:
        x, y = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        x, y = next(train_iter)
    lr_step = get_lr(step, warmup_steps, max_steps, lr)
    for g in optimizer.param_groups:
        g["lr"] = lr_step
    model.train()
    x, y = x.to(device), y.to(device)
    _, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    train_losses.append(loss.item())
    if step == 0:
        print(f"step 0  initial_train_loss={loss.item():.4f}")

    if (step + 1) % eval_interval == 0:
        model.eval()
        val_loss_sum, val_n = 0.0, 0
        with torch.no_grad():
            for x, y in DataLoader(val_ds, batch_size=batch_size):
                x, y = x.to(device), y.to(device)
                _, l = model(x, y)
                val_loss_sum += l.item() * x.size(0)
                val_n += x.size(0)
        val_loss = val_loss_sum / val_n
        val_losses.append(val_loss)
        print(f"step {step+1}/{max_steps}  train_loss={train_losses[-1]:.4f}  val_loss={val_loss:.4f}  lr={lr_step:.2e}")
    if (step + 1) % 500 == 0:
        torch.save({"step": step+1, "model": model.state_dict(), "optimizer": optimizer.state_dict()}, "gpt_shakespeare_ckpt.pt")

print("Training done.")

### Loss tracking — plot (train & validation)

Below we plot **train loss** (every step) and **validation loss** (every 200 steps) so We can see the learning curve at a glance. Train loss should decrease; val loss should follow and then level off (or rise if overfitting).

In [ ]:
import matplotlib.pyplot as plt

# Train loss: one value per step
plt.figure(figsize=(10, 5))
plt.plot(range(len(train_losses)), train_losses, label="Train loss", alpha=0.8, linewidth=0.5)
# Val loss: logged every eval_interval steps (200, 400, 600, ...)
val_steps = [eval_interval * (i + 1) for i in range(len(val_losses))]
plt.plot(val_steps, val_losses, label="Val loss", color="orange", linewidth=2, marker="o", markersize=3)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.legend()
plt.title("Training and validation loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Sampling — Autoregressive Generation

We implement a **`generate`** function that produces **at least 100–200 characters** of "fake" Shakespeare as required:

- **Input:** A starting string (e.g. `"To be or not to "`) and the trained model.
- **Process:** Encode the prompt to token IDs. Then **autoregressively**: take the last `block_size` tokens, run the model, take the logits for the **next** token, sample from the softmax (with a **temperature** to control randomness), append that token to the sequence, and repeat. Because the model is **causal**, it only ever sees the past — so this is valid left-to-right generation.
- **Output:** Decode the sequence of token IDs back to characters and return the string (prompt + generated part). We use `max_new=200` so We can see at least 200 generated characters.

In [ ]:
# Autoregressive generation: prompt -> encode -> repeatedly predict next token (causal), sample, append; decode to string.
def generate(model, itos, stoi, device, start="To be or not to ", max_new=200, temperature=0.8):
    model.eval()
    idx = torch.tensor([[stoi.get(c, 0) for c in start]], device=device)
    for _ in range(max_new):
        idx_cond = idx[:, -model.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)
    return "".join(itos[i] for i in idx[0].tolist())

print(generate(model, itos, stoi, device, start="To be or not to ", max_new=200, temperature=0.8))